# 1. Set Paths

In [ ]:
import os
import json
from datetime import datetime

In [2]:
DICOM_PATH = '/data/dpt_mrincoming-nmr/2024'

DATASET_PATH = '/data/pt_03002/data'

SOURCE_PATH = os.path.join(DATASET_PATH, "source")
PREPARED_PATH = os.path.join(DATASET_PATH, "temp")
BIDSIFIED_PATH = os.path.join(DATASET_PATH, "bids")
RESOURCES_PATH = os.path.join(BIDSIFIED_PATH, "code/resources")
WORKING_DIR = os.getcwd()

In [3]:
### create source directories
# Check if the directories exist, if not, create them
if not os.path.exists(SOURCE_PATH):
    os.makedirs(SOURCE_PATH)

ids_dates = os.path.join(DATASET_PATH, 'ids_dates.json')
with open(ids_dates, 'r') as f:
    ids_dates_data = json.load(f)

subject_ids = list(ids_dates_data.keys())
session_ids = [datetime.strptime(date, "%d.%m.%Y").strftime("%Y%m%d") for date in ids_dates_data.values()]

print("Subject IDs:", subject_ids)
print("Session IDs:", session_ids)


Subject IDs: ['15484.08', '16690.98', '28392.ca', '35080.16', '36623.fd', '37624.ab', '38821.00', '38971.74', '39101.51', '40181.e2']
Session IDs: ['20241107', '20241105', '20241119', '20241112', '20241107', '20241119', '20241119', '20241112', '20241107', '20241112']


In [4]:
### populate source directory

run_dcm2niix = True ## time-consuming!!!

for num, subID in enumerate(subject_ids):
    subDIR = os.path.join(SOURCE_PATH, subID)
    if not os.path.exists(subDIR):
        os.makedirs(subDIR)
    else:
        print(f"Subject directory {subID} already exists in {SOURCE_PATH}")

    sesDIR = os.path.join(subDIR, session_ids[num])
    if not os.path.exists(sesDIR):
        os.makedirs(sesDIR)
        
        ## in DICOM_PATH look for matching subject directory and session directory
        dicom_sub_path = os.path.join(DICOM_PATH, subID)
        if os.path.exists(dicom_sub_path):
            session_dirs = [d for d in os.listdir(dicom_sub_path) if d.startswith(session_ids[num][2:])] # session directory names are in the form of YYMMDD_HHMMSS
            if len(session_dirs) == 1:
                dicom_session_path = os.path.join(dicom_sub_path, session_dirs[0]) + '/'
                os.system(f'ln -s {dicom_session_path} {os.path.join(sesDIR, "dcm")}')
            else:
                # raise a warning if there are multiple sessions for the same date
                raise ValueError(f"Expected exactly one session directory for subject '{subID}' on session '{session_ids[num]}', but found {len(session_dirs)}")
        else:
            raise FileNotFoundError(f"Subject directory {dicom_sub_path} does not exist in DICOM_PATH")
    else: 
        print(f"Session directory {session_ids[num]} already exists in subject '{subID}'")

    if run_dcm2niix: # convert DICOM to NIfTI using dcm2niix (time-consuming)
        nifti_dir = os.path.join(sesDIR, "nii_dcm2niix")
        if not os.path.exists(nifti_dir):
            os.makedirs(nifti_dir)
        
        print(f"""
            -----------------------------------------
            Converting DICOM to NIfTI for subject '{subID}' session '{session_ids[num]}'
            """)

        # only run conversion if the session directory is empty 
        if not os.listdir(nifti_dir): 
            os.system(f'dcm2niix -o {nifti_dir} -ba y -b y -f %p_%4s/s%t-%e -z n {os.path.join(sesDIR, "dcm")}')
            print(f"""
            Conversion completed for subject '{subID}' session '{session_ids[num]}'
            =========================================
            """)
        else:
            print(f"""
            Conversion canceled, because session directory {sesDIR} is not empty
            =========================================
            """)


Subject directory 15484.08 already exists in /data/pt_03002/data/source
Session directory 20241107 already exists in subject '15484.08'

            -----------------------------------------
            Converting DICOM to NIfTI for subject '15484.08' session '20241107'
            

            Conversion canceled, because session directory /data/pt_03002/data/source/15484.08/20241107 is not empty
            
Subject directory 16690.98 already exists in /data/pt_03002/data/source
Session directory 20241105 already exists in subject '16690.98'

            -----------------------------------------
            Converting DICOM to NIfTI for subject '16690.98' session '20241105'
            

            Conversion canceled, because session directory /data/pt_03002/data/source/16690.98/20241105 is not empty
            
Subject directory 28392.ca already exists in /data/pt_03002/data/source
Session directory 20241119 already exists in subject '28392.ca'

            ---------------------

In [5]:
print("Dataset path:", os.path.isdir(DATASET_PATH))
print("Source path:", os.path.isdir(SOURCE_PATH))
print("Prepared path:", os.path.isdir(PREPARED_PATH))
print("Bidsified path:", os.path.isdir(BIDSIFIED_PATH))
print("Resources path:", os.path.isdir(RESOURCES_PATH))
print("Working directory:", WORKING_DIR)

Dataset path: True
Source path: True
Prepared path: True
Bidsified path: True
Resources path: True
Working directory: /data/u_kuegler_software/git/MPM_bidsification


# 2. Initialize `bidsme` and get the `logger` object
which which will control the logging of all bidsme functions:will control the logging of all bidsme functions:

In [6]:
import bidsme
logger = bidsme.init()

main(81) - INFO 
main(82) - INFO -------------- START bidsme ----------------
main(83) - INFO Mon Dec  9 17:04:45 2024
main(84) - INFO version: 1.8.1
bidsme.schema.BIDSschema(670) - INFO Loaded BIDS schema version 1.10.0


# 3. Prepare data set for bidsification

In [ ]:
# help(bidsme.prepare)

Help on function prepare in module bidsme.prepare:

prepare(
    source: str,
    destination: str,
    plugin_file: str = '',
    plugin_opt: dict = {},
    sub_list: list = [],
    sub_skip_tsv: bool = False,
    sub_skip_dir: bool = False,
    ses_skip_dir: bool = False,
    part_template: str = '',
    sub_prefix: str = '',
    ses_prefix: str = '',
    sub_no_dir: bool = False,
    ses_no_dir: bool = False,
    data_dirs: dict = {},
    dry_run: bool = False
) -> None
    Prepare data from surce folder and place it in
    sestination folder.

    Source folder is expected to have structure
    source/[subId/][sesId/][data/]file.
    Absence of subId and sesId levels must be communicated
    via sub_no_dir and ses_no_dir options. List of data
    folders must be given in data_dirs.

    Prepeared data will have structure
    destination/sub-<subId>/ses-<sesId>/<type>/<sequence>/file

    A list of treated subjects will be created/updated
    in destination/participants.tsv file

  

In [ ]:
logger.setLevel("INFO")
bidsme.prepare(SOURCE_PATH, PREPARED_PATH, 
               data_dirs={"nii_dcm2niix/localizer*":"MRI",
                          "nii_dcm2niix/ThreeDream_5mm_40deg*":"MRI",
                          "nii_dcm2niix/smap_kp_mtflash3d_*_4p0*":"MRI",
                          "nii_dcm2niix/t1w_kp_mtflash3d_*_0p6*":"MRI",
                          "nii_dcm2niix/pdw_kp_mtflash3d_v1s_0p6*":"MRI",
                          "nii_dcm2niix/mtw_kp_mtflash3d_v1s_0p6*":"MRI",
                          "nii_dcm2niix/kp_afib1*":"MRI",
                          "nii_dcm2niix/t1_mp2rage_sag_*":"MRI",},
                plugin_file=os.path.join(RESOURCES_PATH, "plugins/plugin_prepare_nk.py"),
                part_template=os.path.join(RESOURCES_PATH,"table_templates/participants_nk.json"),
                # sub_list=['sub-008']
              )
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

bidsme.prepare(192) - INFO -------------- Prepearing data -------------
bidsme.prepare(193) - INFO Source directory: /data/pt_03002/data/source
bidsme.prepare(194) - INFO Destination directory: /data/pt_03002/data/temp
bidsme.plugins.plugins(79) - INFO Loading module plugin_prepare_nk from /data/pt_03002/data/bids/code/resources/plugins/plugin_prepare_nk.py
Loading sessions_nk.json from /data/pt_03002/data/bids/code/resources/table_templates/sessions_nk.json.
          This functionality is not part of Bidsme, but implemented in a plugin. 
          It only works for processing all sessions of a subjects. Problems may 
          arise if the plugin is used for single sessions.
bidsme.bidsMeta.BidsTable(134) - INFO Loaded participants.tsv table with 10 entries
bidsme.prepare(280) - INFO Skipping subject 'sub-005'
bidsme.prepare(280) - INFO Skipping subject 'sub-002'
bidsme.prepare(280) - INFO Skipping subject 'sub-011'
bidsme.prepare(280) - INFO Skipping subject 'sub-009'
bidsme.prepare

> **Note**: **apparently, it is not possible to specify a specific subset of sessions**
> + If the user wishes to rename subjects and/or sessions, it can be done with plug-in functions ```SubjectEP``` and ```SessionEP``` or by renaming directly folders in the prepared dataset.

# 4. Create the bidsmap.yaml

+ most tedious part of the process

In [75]:
# help(bidsme.mapper)

Help on function mapper in module bidsme.mapper:

mapper(
    source: str,
    destination: str,
    plugin_file: str = '',
    plugin_opt: dict = {},
    sub_list: list = [],
    sub_skip_tsv: bool = False,
    sub_skip_dir: bool = False,
    ses_skip_dir: bool = False,
    process_all: bool = False,
    bidsmapfile: str = 'bidsmap.yaml',
    map_template: str = 'bidsmap_template.yaml',
    dry_run: bool = False
) -> None
    Generates bidsmap.yaml from prepeared dataset and
    map template.

    Only subjects in source/participants.tsv are treated,
    this list can be narrowed using sub_list, sub_skip_tsv
    and sub_skip_dir options

    Parameters
    ----------
    source: str
        folder containing source dataset
    destination: str
        folder for prepeared dataset
    plugin_file: str
        path to the plugin file to use
    plugin_opt: dict
        named options passed to plugin
    sub_list: list
        list of subject to process. Subjects
        are checked afte

In [20]:
PLUGIN_BIDS=os.path.join(RESOURCES_PATH, "plugins/plugin_bidsify_nk.py")

In [21]:
bidsme.mapper(PREPARED_PATH, BIDSIFIED_PATH, plugin_file=PLUGIN_BIDS)
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

bidsme.mapper(247) - INFO ------------ Generating bidsmap ------------
bidsme.mapper(248) - INFO Current directory: /data/u_kuegler_software/git/MPM_bidsification
bidsme.mapper(249) - INFO Source directory: /data/pt_03002/data/temp
bidsme.mapper(250) - INFO Destination directory: /data/pt_03002/data/bids
bidsme.mapper(266) - INFO loading template bidsmap bidsmap_template.yaml
bidsme.mapper(285) - INFO loading working bidsmap /data/pt_03002/data/bids/code/bidsme/bidsmap.yaml
bidsme.plugins.plugins(79) - INFO Loading module plugin_bidsify_nk from /data/pt_03002/data/bids/code/resources/plugins/plugin_bidsify_nk.py
bidsme.bidsMeta.BidsTable(134) - INFO Loaded participants.tsv table with 10 entries
bidsme.bidsMeta.BidsTable(141) - INFO Created empty participants.tsv table
bidsme.mapper(385) - INFO sub-002 (1/10): Scanning folder /data/pt_03002/data/temp/sub-002/ses-01
bidsme.mapper(63) - INFO Processing: sub 'sub-002', ses 'ses-01', 001-localizer/0 (1 files)
bidsme.mapper(63) - INFO Proces

+ open the created yaml file in VS Code
+ fix each warning/error, save the file, and repeat the code block above
    - find help at in the [Jupyter Notebooks in the bidsme tutorial](https://github.com/CyclotronResearchCentre/bidsme_tutorial) or in the docs [under Bidsmap creation](https://github.com/CyclotronResearchCentre/bidsme/blob/dev/doc/creating_map.md)
+ resume until there are no warnings left

> **Note:** to find a specific string in a file, use the command ```cat file.json | grep -i "string"``` **or** use ```less file.json``` and search using ```/string``` (```n``` will take you to the next entry & ```shift+n``` to the previous one; ```-I``` for case-insensitive)

> **Note:** Bidsme allow some limited transformation of data retrieved from header, these transformations are called actions and are defined in function ```action_value``` in file ```INSTALLATION_PATH/bidsme/Modules/common.py```.

```
Accepted actions:
    "": no action, return value
    int: cast value to int
    float: cast value to float
    str: cast value to string
    format<parameters>: apply python3 formatting
        mini-language to value, {:<parameters>}.format(value)
    scale<int>: apply a 10-based scale to value,
        value ** <int>
    mult<float>: multiply value
    div<float>: divide value
    round<int>: round value to given precision
```


+ The naming schema and sidecar json fields for a given modality (in this case MRI) are defined in $INSTALLATION_PATH/bidsme/Modules/MRI/_MRI.py. The list of entities is stored in modalities dictionary. If an image belongs for example to anat, bidsme will load the list of entities from modalities["anat"].

+ The optional model field will foce to use different list of entities from modalities dictionary. We will use the models extensively, while creating map for MPM part of the examle dataset.


# 5. Bidsification of the data set

In [ ]:
# !bidsme bidsify -help

usage: bidsme bidsify [-q] [--level {DEBUG,INFO,WARNING,ERROR,DEBUG}]
                      [--formatter FORMATTER] [--plugin PLUGIN]
                      [-o Name=Value [Name=Value ...]]
                      [--participants ID [ID ...]] [--skip-in-tsv]
                      [--skip-existing] [--skip-existing-sessions] [--dry-run]
                      [--part-template TEMPLATE] [-b BIDSMAP] [-h]
                      source destination

Bidsification of dataset

positional arguments:
  source                Path to the source dataset
  destination           Path to the destination dataset

options:
  --dry-run             Run in dry mode, i.e. without writting anything on the
                        disk (default: False)

logging options:
  -q, --quiet           Silence stdout logging output (default: False)
  --level {DEBUG,INFO,WARNING,ERROR,DEBUG}
                        Set logging level (default: INFO)
  --formatter FORMATTER
                        Logging formatting string (d

In [17]:
MAP_FILE = os.path.join(BIDSIFIED_PATH, "code/bidsme/bidsmap.yaml")
PLUGIN_BIDS=os.path.join(RESOURCES_PATH, "plugins/plugin_bidsify_nk.py")

-b = mapping file, --plugin = plugin

In [18]:
!bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_BIDS
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_BIDS --participants 'sub-008' 'sub-009'
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_BIDS --skip-existing

main(81) - INFO 
main(82) - INFO -------------- START bidsme ----------------
main(83) - INFO Mon Dec  9 17:20:05 2024
main(84) - INFO version: 1.8.1
bidsme.schema.BIDSschema(670) - INFO Loaded BIDS schema version 1.10.0
bidsme.bidsify(188) - INFO -------------- Prepearing data -------------
bidsme.bidsify(189) - INFO Source directory: /data/pt_03002/data/temp
bidsme.bidsify(190) - INFO Destination directory: /data/pt_03002/data/bids
bidsme.bidsify(212) - WARNING Dataset description file 'dataset_description.json' not found in '/data/pt_03002/data/bids'
bidsme.bidsify(218) - WARNING Dataset readme file 'README' not found in '/data/pt_03002/data/bids'
bidsme.bidsify(233) - INFO loading bidsmap /data/pt_03002/data/bids/code/bidsme/bidsmap.yaml
bidsme.plugins.plugins(79) - INFO Loading module plugin_bidsify_nk from /data/pt_03002/data/bids/code/resources/plugins/plugin_bidsify_nk.py
bidsme.bidsMeta.BidsTable(134) - INFO Loaded participants.tsv table with 10 entries
bidsme.bidsMeta.BidsTab

## Hints


The `004-al_mtflash3d_PDw` and `005-al_mtflash3d_PDw` are
the anatomical images
(suffix -- `MPM`)
taken using several echo times (`echo-1` ... `echo-6`),
and splitted into magnitude and phase components (`part-mag` and `part-phase`).
Additionaly, as for the PD-weighted images, the MT pulse wasn't used, we will
add the `mt-off` entity.
We will also add `flip-1` to the name, to mark that PDw images uses different
flip angle from T
So the final name will become:
`anat/sub-001_ses-s01530_acq-PDw_echo-1_mt-off_part-mag_MPM.nii`.
`anat/sub-001_ses-s01530_acq-PDw_echo-1_flip-1_mt-off_part-mag_MPM.nii`

The `002-al_mtflash3d_sensArray` and `003-al_mtflash3d_sensBody`
are [B1 fieldmaps](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#rb1cor-specific-notes)
(suffix -- `RB1COR`),
taken for the PD-weighted images, using head and body coils
(`acq-headPDw` and `acq-bodyPDw`).
So their names will be simply: `fmap/sub-001_ses-s01530_acq-headPDw_RB1COR.nii`.

The similar names can be applied to T1w and MTw images and corresponding fieldmaps,
using corresponding `acq-` entities `acq-T1w` and `acq-MTw`.
For MTw images we alse need to use the `mt-on` entity.

Finally, `014-al_B1mapping` is the
[global B1 map](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#tb1epi-specific-notes)
(suffix -- `TB1EPI`)
sets of images, taken with two echo times (`echo-1`, `echo-2`)
and several flip angles (`flip-01`, ... `flip-08`).
So the final name will become: `fmap/sub-001_ses-s01530_echo-1_flip-01_TB1EPI`